# CMIP7 API example

This notebook uses a tiny synthetic CMIP7 dataset to show the public Woodpecker API: check a dataset, dry-run fixes, apply fixes in memory, and re-check the result.

In [ ]:
import woodpecker_cmip7_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_cmip7

Create a deterministic CMIP7-like dataset with two common issues: missing `project_id` metadata and a legacy `temp` variable name.

In [ ]:
dataset = make_cmip7(
    missing=["project_id"],
    rename_vars={"tas": "temp"},
    seed=7,
)

dataset

In [ ]:
fix_ids = [
    "cmip7.ensure_project_id_present",
    "cmip7.rename_temp_variable_to_tas",
]

findings = woodpecker.check(dataset, identifiers=fix_ids)
findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
dry_run = woodpecker.fix(dataset, identifiers=findings.fix_ids, write=False)

dry_run.stats, tuple(dataset.data_vars), dataset.attrs.get("project_id")

Apply the same fixes in memory with `write=True`.

In [ ]:
write = woodpecker.fix(dataset, identifiers=findings.fix_ids, write=True)

write.stats, tuple(dataset.data_vars), dataset.attrs.get("project_id")

In [ ]:
recheck = woodpecker.check(dataset, identifiers=fix_ids)
recheck.has_findings